# EP22. 회의록 → 액션 아이템 에이전트

## 학습 목표

1. **Whisper API**로 한국어 회의 음성을 텍스트로 변환하고, 텍스트 폴백을 구현한다
2. **Pydantic 구조화 출력**으로 회의록에서 액션 아이템을 정형 데이터로 추출한다 (EP08 심화)
3. **FastMCP Mock 서버**로 Jira/Notion/Slack 연동을 안전하게 시뮬레이션한다 (EP12 활용)
4. **LangGraph 멀티에이전트 파이프라인**으로 요약→추출→등록 전체 흐름을 자동화한다 (EP10 패턴)

---

### AI 가이드

이 노트북은 30분 회의에서 자동으로 Jira 티켓 5개를 생성하는 에이전트를 구축합니다.  
음성→텍스트→요약→구조화 추출→외부 시스템 등록의 전체 파이프라인을 다룹니다.

### 사전 요구사항

| 에피소드 | 내용 | 활용 |
|----------|------|------|
| **EP08** 구조화 출력 | Pydantic + `with_structured_output` | 액션 아이템 추출 |
| **EP10** 멀티에이전트 | LangGraph StateGraph 패턴 | 파이프라인 구성 |
| **EP12** MCP 서버 | FastMCP 서버 구축 | 외부 시스템 Mock 연동 |

**예상 소요 시간**: 80분  
**필요 API 키**: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY` (Langfuse 선택: `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`)

## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install openai fastmcp langgraph langchain langchain-anthropic langfuse pydantic python-dotenv --quiet

## 섹션 2. 라이브러리 로드 + API 키

In [ ]:
import os
import re
import json
from datetime import datetime, timedelta
from enum import Enum
from typing import Optional

from dotenv import load_dotenv

load_dotenv()

# 필수 API 키 확인
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요."
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요."

print("필수 API 키 확인 완료")

# 선택적 Langfuse 키
LANGFUSE_ENABLED = all([
    os.getenv("LANGFUSE_PUBLIC_KEY"),
    os.getenv("LANGFUSE_SECRET_KEY")
])
print(f"Langfuse 활성화: {LANGFUSE_ENABLED}")

## 섹션 3. Whisper API 음성 → 텍스트 (텍스트 폴백 포함)

실제 음성 파일이 없는 경우, `data/sample_meeting.txt`를 텍스트 폴백으로 로드합니다.

In [ ]:
from openai import OpenAI
from pathlib import Path

client = OpenAI()

# --- Whisper API 음성→텍스트 함수 ---
def transcribe_audio(audio_path: str, language: str = "ko") -> str:
    """Whisper API로 음성 파일을 텍스트로 변환합니다."""
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            language=language,
            response_format="verbose_json",
            timestamp_granularities=["segment"]
        )
    # 세그먼트를 텍스트로 결합
    full_text = "\n".join(
        f"[{seg.start:.1f}s] {seg.text}" for seg in transcript.segments
    )
    return full_text


# --- 텍스트 폴백: sample_meeting.txt 로드 ---
def load_text_fallback(text_path: str) -> str:
    """텍스트 파일에서 회의록을 로드합니다 (음성 파일이 없을 때 사용)."""
    with open(text_path, "r", encoding="utf-8") as f:
        return f.read()


# --- 메인 로직: 음성 파일이 있으면 Whisper, 없으면 텍스트 폴백 ---
AUDIO_FILE = "meeting.wav"  # 음성 파일 경로 (있으면 사용)
TEXT_FALLBACK = "../../data/sample_meeting.txt"  # 텍스트 폴백 경로

if Path(AUDIO_FILE).exists():
    print(f"음성 파일 발견: {AUDIO_FILE}")
    print("Whisper API로 변환 중...")
    raw_transcript = transcribe_audio(AUDIO_FILE)
    print(f"변환 완료: {len(raw_transcript)}자")
else:
    print(f"음성 파일 없음 → 텍스트 폴백 사용: {TEXT_FALLBACK}")
    raw_transcript = load_text_fallback(TEXT_FALLBACK)
    print(f"텍스트 로드 완료: {len(raw_transcript)}자")

print("\n--- 회의록 미리보기 (처음 500자) ---")
print(raw_transcript[:500])

## 섹션 4. 텍스트 전처리 (화자 분리, 불필요한 부분 제거)

In [ ]:
def preprocess_transcript(raw_text: str) -> dict:
    """회의록 텍스트를 전처리합니다.
    
    Returns:
        dict: {
            'metadata': {...},
            'utterances': [{'speaker': str, 'text': str}, ...],
            'full_text': str  # 화자:발화 형식의 정리된 텍스트
        }
    """
    lines = raw_text.strip().split("\n")
    
    # 1. 메타데이터 추출
    metadata = {}
    content_lines = []
    
    for line in lines:
        stripped = line.strip()
        if stripped.startswith("[회의록]"):
            metadata["title"] = stripped.replace("[회의록] ", "")
        elif stripped.startswith("일시:"):
            metadata["datetime"] = stripped.replace("일시: ", "")
        elif stripped.startswith("장소:"):
            metadata["location"] = stripped.replace("장소: ", "")
        elif stripped.startswith("참석자:"):
            metadata["participants"] = stripped.replace("참석자: ", "")
        elif stripped.startswith("회의 종료:"):
            metadata["end_time"] = stripped.replace("회의 종료: ", "")
        elif stripped and stripped != "---":
            content_lines.append(stripped)
    
    # 2. 화자:발화 파싱
    utterances = []
    speaker_pattern = re.compile(r"^(.+?):\s*(.+)$")
    
    for line in content_lines:
        match = speaker_pattern.match(line)
        if match:
            speaker = match.group(1).strip()
            text = match.group(2).strip()
            utterances.append({"speaker": speaker, "text": text})
    
    # 3. 화자 통계
    from collections import Counter
    speaker_counts = Counter(u["speaker"] for u in utterances)
    metadata["speaker_stats"] = dict(speaker_counts)
    
    # 4. 정리된 전체 텍스트
    full_text = "\n".join(
        f"{u['speaker']}: {u['text']}" for u in utterances
    )
    
    return {
        "metadata": metadata,
        "utterances": utterances,
        "full_text": full_text
    }

In [ ]:
# 전처리 실행
processed = preprocess_transcript(raw_transcript)

print("=== 메타데이터 ===")
for key, value in processed["metadata"].items():
    print(f"  {key}: {value}")

print(f"\n=== 발화 수: {len(processed['utterances'])}개 ===")
print("\n=== 처음 5개 발화 ===")
for u in processed["utterances"][:5]:
    print(f"  [{u['speaker']}] {u['text'][:60]}...")

## 섹션 5. 회의록 요약 에이전트

In [ ]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="claude-sonnet-4-20250514",
    temperature=0,
    max_tokens=4096
)

SUMMARY_PROMPT = """당신은 전문 회의록 요약 에이전트입니다.
다음 회의록을 분석하여 구조화된 요약을 생성하세요.

## 회의 정보
- 제목: {title}
- 일시: {datetime}
- 참석자: {participants}

## 회의록
{transcript}

## 출력 형식 (반드시 아래 형식을 따르세요)

### 한줄 요약
(30자 이내로 회의 핵심을 요약)

### 주요 논의 사항
1. (첫 번째 논의 사항)
2. (두 번째 논의 사항)
...

### 주요 수치/데이터
- (회의 중 언급된 구체적 수치와 데이터)

### 핵심 결정 사항
- (합의된 결정 사항)

### 미결 사항
- (추후 논의가 필요한 사항)

### 다음 회의 안건
- (다음 회의에서 다룰 내용)
"""

print("요약 에이전트 프롬프트 준비 완료")

In [ ]:
# 요약 실행
summary_input = SUMMARY_PROMPT.format(
    title=processed["metadata"].get("title", "제목 없음"),
    datetime=processed["metadata"].get("datetime", "일시 미상"),
    participants=processed["metadata"].get("participants", "참석자 미상"),
    transcript=processed["full_text"]
)

summary_response = llm.invoke(summary_input)
meeting_summary = summary_response.content

print("=== 회의록 요약 ===")
print(meeting_summary)

## 섹션 6. 액션 아이템 추출 (Pydantic 구조화 출력)

EP08에서 배운 `with_structured_output`을 활용하여 회의록에서 액션 아이템을 추출합니다.

In [ ]:
from pydantic import BaseModel, Field


class Priority(str, Enum):
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"


class Category(str, Enum):
    DEVELOPMENT = "development"
    DESIGN = "design"
    MARKETING = "marketing"
    PLANNING = "planning"
    RESEARCH = "research"


class ActionItem(BaseModel):
    """회의에서 추출된 개별 액션 아이템"""
    assignee: str = Field(description="담당자 이름 (예: 이개발)")
    description: str = Field(description="구체적인 업무 내용")
    deadline: str = Field(description="마감일 (YYYY-MM-DD 형식)")
    priority: Priority = Field(description="우선순위: high/medium/low")
    category: Category = Field(description="업무 카테고리")
    context: str = Field(description="회의 중 해당 액션 아이템이 언급된 맥락 요약")


class MeetingActions(BaseModel):
    """회의 전체의 액션 아이템 모음"""
    meeting_title: str = Field(description="회의 제목")
    meeting_date: str = Field(description="회의 일시 (YYYY-MM-DD 형식)")
    action_items: list[ActionItem] = Field(description="추출된 액션 아이템 목록")
    key_decisions: list[str] = Field(description="핵심 결정 사항 목록")


print("Pydantic 모델 정의 완료")
print(f"ActionItem 필드: {list(ActionItem.model_fields.keys())}")
print(f"MeetingActions 필드: {list(MeetingActions.model_fields.keys())}")

In [ ]:
# 구조화 출력으로 액션 아이템 추출
structured_llm = llm.with_structured_output(MeetingActions)

EXTRACTION_PROMPT = """당신은 회의록에서 액션 아이템을 정확하게 추출하는 전문 에이전트입니다.

다음 회의록에서 모든 액션 아이템을 추출하세요.
각 액션 아이템은 반드시 다음을 포함해야 합니다:
- assignee: 담당자 이름 (회의록에 명시된 이름)
- description: 구체적인 업무 내용 (무엇을 해야 하는지 명확하게)
- deadline: 마감일 (YYYY-MM-DD 형식, 회의록에 명시된 날짜)
- priority: 우선순위 (업무의 중요도와 긴급도에 따라 판단)
- category: 카테고리 (development/design/marketing/planning/research 중 선택)
- context: 회의 중 해당 업무가 언급된 맥락 요약

"~하겠습니다", "~까지", "~완료" 등의 표현에서 액션 아이템을 식별하세요.

## 회의록
{transcript}
"""

extraction_result = structured_llm.invoke(
    EXTRACTION_PROMPT.format(transcript=processed["full_text"])
)

print(f"=== 추출 결과 ===")
print(f"회의 제목: {extraction_result.meeting_title}")
print(f"회의 일자: {extraction_result.meeting_date}")
print(f"추출된 액션 아이템: {len(extraction_result.action_items)}개")
print(f"핵심 결정 사항: {len(extraction_result.key_decisions)}개")

In [ ]:
# 추출된 액션 아이템 상세 출력
print("=== 추출된 액션 아이템 상세 ===")
for i, item in enumerate(extraction_result.action_items, 1):
    priority_emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}
    emoji = priority_emoji.get(item.priority.value, "⚪")
    print(f"\n{emoji} 아이템 {i}")
    print(f"   담당자: {item.assignee}")
    print(f"   업무: {item.description}")
    print(f"   기한: {item.deadline}")
    print(f"   우선순위: {item.priority.value}")
    print(f"   카테고리: {item.category.value}")
    print(f"   맥락: {item.context}")

print("\n=== 핵심 결정 사항 ===")
for i, decision in enumerate(extraction_result.key_decisions, 1):
    print(f"  {i}. {decision}")

## 섹션 7. Mock MCP 서버 (Jira / Notion 시뮬레이션)

EP12에서 배운 FastMCP를 활용하여 외부 시스템을 Mock으로 구현합니다.  
실제 API를 호출하지 않고 `print`로 결과를 출력하여 안전하게 테스트합니다.

In [ ]:
from fastmcp import FastMCP

# ==========================================
# Mock Jira MCP 서버
# ==========================================
jira_server = FastMCP("Jira Mock Server")

_jira_tickets = []  # 생성된 티켓 저장소


@jira_server.tool
def create_ticket(
    title: str,
    description: str,
    assignee: str,
    priority: str,
    deadline: str,
    category: str
) -> dict:
    """Jira에 새 티켓을 생성합니다."""
    ticket_id = f"PROJ-{len(_jira_tickets) + 101:03d}"
    ticket = {
        "ticket_id": ticket_id,
        "title": title,
        "description": description,
        "assignee": assignee,
        "priority": priority,
        "deadline": deadline,
        "category": category,
        "status": "To Do",
        "created_at": datetime.now().isoformat(),
        "url": f"https://jira.example.com/browse/{ticket_id}"
    }
    _jira_tickets.append(ticket)
    print(f"  [Jira] 티켓 생성됨: {ticket_id} | {title}")
    print(f"         담당: {assignee} | 기한: {deadline} | 우선순위: {priority}")
    return ticket


@jira_server.tool
def list_tickets() -> list[dict]:
    """생성된 모든 Jira 티켓을 조회합니다."""
    return _jira_tickets


print(f"Jira Mock Server 초기화 완료 (도구 수: 2)")

In [ ]:
# ==========================================
# Mock Notion MCP 서버
# ==========================================
notion_server = FastMCP("Notion Mock Server")

_notion_pages = []  # 생성된 페이지 저장소


@notion_server.tool
def create_meeting_page(
    title: str,
    summary: str,
    action_items_json: str,
    participants: str
) -> dict:
    """Notion에 회의록 페이지를 생성합니다."""
    page_id = f"page-{len(_notion_pages) + 1:03d}"
    page = {
        "page_id": page_id,
        "title": title,
        "summary": summary,
        "action_items": action_items_json,
        "participants": participants,
        "created_at": datetime.now().isoformat(),
        "url": f"https://notion.so/{page_id}"
    }
    _notion_pages.append(page)
    print(f"  [Notion] 페이지 생성됨: {page_id} | {title}")
    print(f"           요약 길이: {len(summary)}자 | 액션 아이템 포함")
    return page


print(f"Notion Mock Server 초기화 완료 (도구 수: 1)")

In [ ]:
# ==========================================
# Mock Slack MCP 서버
# ==========================================
slack_server = FastMCP("Slack Mock Server")

_slack_messages = []  # 전송된 메시지 저장소


@slack_server.tool
def send_message(channel: str, message: str) -> dict:
    """Slack 채널에 메시지를 전송합니다."""
    msg = {
        "channel": channel,
        "message": message,
        "sent_at": datetime.now().isoformat(),
        "status": "sent"
    }
    _slack_messages.append(msg)
    print(f"  [Slack] #{channel}에 메시지 전송됨")
    print(f"          내용: {message[:80]}..." if len(message) > 80 else f"          내용: {message}")
    return msg


print(f"Slack Mock Server 초기화 완료 (도구 수: 1)")

In [ ]:
# ==========================================
# Mock MCP 서버 테스트: 직접 호출
# ==========================================
# 노트북 환경에서는 MCP 서버를 직접 함수로 호출하여 테스트합니다.

print("=== Mock MCP 서버 직접 호출 테스트 ===\n")

# Jira 티켓 생성 테스트
test_ticket = create_ticket(
    title="검색 API 최적화 PoC",
    description="Elasticsearch 인덱스 재설계를 통해 검색 API 응답 속도를 400ms 이하로 개선",
    assignee="이개발",
    priority="high",
    deadline="2026-04-15",
    category="development"
)
print(f"  → 반환값: {test_ticket['ticket_id']}\n")

# Notion 페이지 생성 테스트
test_page = create_meeting_page(
    title="Q2 제품 전략 회의 회의록",
    summary="Q2 검색 기능 고도화 집중 결정",
    action_items_json=json.dumps([{"assignee": "이개발", "task": "검색 API PoC"}], ensure_ascii=False),
    participants="김팀장, 이개발, 박기획, 최마케, 정디자"
)
print(f"  → 반환값: {test_page['page_id']}\n")

# Slack 메시지 전송 테스트
test_msg = send_message(
    channel="product-team",
    message="Q2 제품 전략 회의 완료! 액션 아이템 5건이 Jira에 등록되었습니다."
)
print(f"  → 전송 상태: {test_msg['status']}")

# 저장소 초기화 (테스트 데이터 제거)
_jira_tickets.clear()
_notion_pages.clear()
_slack_messages.clear()
print("\n테스트 완료. 저장소 초기화됨.")

## 섹션 8. 멀티에이전트 파이프라인 (LangGraph)

EP10에서 배운 LangGraph StateGraph 패턴으로 요약→추출→등록 파이프라인을 구성합니다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated


# ==========================================
# 상태 정의
# ==========================================
class MeetingState(TypedDict):
    # 입력
    transcript: str                # 원본 회의록 텍스트
    metadata: dict                 # 회의 메타데이터
    # 요약 에이전트 출력
    summary: str                   # 요약 결과
    # 추출 에이전트 출력
    action_items: list             # ActionItem 딕셔너리 목록
    key_decisions: list            # 핵심 결정 사항
    # 등록 에이전트 출력
    jira_tickets: list             # 생성된 Jira 티켓
    notion_page: dict              # 생성된 Notion 페이지
    slack_message: dict            # 전송된 Slack 메시지


print("MeetingState 정의 완료")
print(f"상태 필드: {list(MeetingState.__annotations__.keys())}")

In [ ]:
# ==========================================
# 노드 1: 요약 에이전트
# ==========================================
def summarize_meeting(state: MeetingState) -> dict:
    """회의록을 요약합니다."""
    print("\n[노드: summarizer] 회의록 요약 시작...")
    
    prompt = SUMMARY_PROMPT.format(
        title=state["metadata"].get("title", "제목 없음"),
        datetime=state["metadata"].get("datetime", "일시 미상"),
        participants=state["metadata"].get("participants", "참석자 미상"),
        transcript=state["transcript"]
    )
    response = llm.invoke(prompt)
    summary = response.content
    
    print(f"[노드: summarizer] 요약 완료 ({len(summary)}자)")
    return {"summary": summary}

In [ ]:
# ==========================================
# 노드 2: 추출 에이전트
# ==========================================
def extract_actions(state: MeetingState) -> dict:
    """회의록에서 액션 아이템을 구조화 추출합니다."""
    print("\n[노드: extractor] 액션 아이템 추출 시작...")
    
    structured = llm.with_structured_output(MeetingActions)
    result = structured.invoke(
        EXTRACTION_PROMPT.format(transcript=state["transcript"])
    )
    
    # Pydantic 모델을 딕셔너리로 변환 (상태 저장용)
    action_dicts = [item.model_dump() for item in result.action_items]
    
    print(f"[노드: extractor] 추출 완료: {len(action_dicts)}개 액션 아이템")
    return {
        "action_items": action_dicts,
        "key_decisions": result.key_decisions
    }

In [ ]:
# ==========================================
# 노드 3: 등록 에이전트
# ==========================================
def register_actions(state: MeetingState) -> dict:
    """액션 아이템을 외부 시스템에 등록합니다."""
    print("\n[노드: registrar] 외부 시스템 등록 시작...")
    
    # 1. Jira 티켓 생성
    tickets = []
    for item in state["action_items"]:
        ticket = create_ticket(
            title=item["description"][:80],
            description=item["description"],
            assignee=item["assignee"],
            priority=item["priority"],
            deadline=item["deadline"],
            category=item["category"]
        )
        tickets.append(ticket)
    
    # 2. Notion 페이지 생성
    page = create_meeting_page(
        title=state["metadata"].get("title", "회의록"),
        summary=state["summary"],
        action_items_json=json.dumps(state["action_items"], ensure_ascii=False),
        participants=state["metadata"].get("participants", "")
    )
    
    # 3. Slack 알림 전송
    ticket_summary = "\n".join(
        f"  - [{t['ticket_id']}] {t['assignee']}: {t['title']} (기한: {t['deadline']})"
        for t in tickets
    )
    slack_msg = send_message(
        channel="product-team",
        message=(
            f"📋 회의록 자동 처리 완료!\n"
            f"회의: {state['metadata'].get('title', '회의록')}\n"
            f"Jira 티켓 {len(tickets)}건 생성:\n{ticket_summary}\n"
            f"Notion: {page['url']}"
        )
    )
    
    print(f"[노드: registrar] 등록 완료: Jira {len(tickets)}건, Notion 1건, Slack 1건")
    return {
        "jira_tickets": tickets,
        "notion_page": page,
        "slack_message": slack_msg
    }

In [ ]:
# ==========================================
# LangGraph 파이프라인 구성
# ==========================================
graph = StateGraph(MeetingState)

# 노드 등록
graph.add_node("summarizer", summarize_meeting)
graph.add_node("extractor", extract_actions)
graph.add_node("registrar", register_actions)

# 엣지 연결: 순차 파이프라인
graph.add_edge(START, "summarizer")
graph.add_edge("summarizer", "extractor")
graph.add_edge("extractor", "registrar")
graph.add_edge("registrar", END)

# 컴파일
pipeline = graph.compile()

print("LangGraph 파이프라인 컴파일 완료")
print("흐름: START → summarizer → extractor → registrar → END")

In [ ]:
# ==========================================
# 파이프라인 실행
# ==========================================
print("=" * 60)
print("멀티에이전트 파이프라인 실행 시작")
print("=" * 60)

# Mock 저장소 초기화
_jira_tickets.clear()
_notion_pages.clear()
_slack_messages.clear()

# 초기 상태 구성
initial_state = {
    "transcript": processed["full_text"],
    "metadata": processed["metadata"],
    "summary": "",
    "action_items": [],
    "key_decisions": [],
    "jira_tickets": [],
    "notion_page": {},
    "slack_message": {}
}

# 실행
final_state = pipeline.invoke(initial_state)

print("\n" + "=" * 60)
print("파이프라인 실행 완료!")
print("=" * 60)

In [ ]:
# ==========================================
# 최종 결과 확인
# ==========================================
print("=== 최종 결과 요약 ===\n")

print(f"1. 요약 길이: {len(final_state['summary'])}자")
print(f"2. 추출된 액션 아이템: {len(final_state['action_items'])}개")
print(f"3. 핵심 결정 사항: {len(final_state['key_decisions'])}개")
print(f"4. Jira 티켓: {len(final_state['jira_tickets'])}건")
print(f"5. Notion 페이지: {final_state['notion_page'].get('url', 'N/A')}")
print(f"6. Slack 알림: {final_state['slack_message'].get('status', 'N/A')}")

print("\n--- 생성된 Jira 티켓 ---")
for t in final_state["jira_tickets"]:
    print(f"  [{t['ticket_id']}] {t['assignee']}: {t['title'][:50]}")
    print(f"           기한: {t['deadline']} | 우선순위: {t['priority']}")

## 섹션 9. 품질 평가 (기대 결과 vs 추출 결과)

EP04 Harness 방식으로 추출된 액션 아이템의 품질을 정량적으로 평가합니다.

In [ ]:
# ==========================================
# 기대 액션 아이템 정의 (정답)
# ==========================================
expected_items = [
    {
        "assignee": "이개발",
        "description": "검색 API 최적화 PoC 완료",
        "deadline": "2026-04-15",
        "category": "development"
    },
    {
        "assignee": "박기획",
        "description": "자동 완성 기능 PRD 문서 작성",
        "deadline": "2026-04-10",
        "category": "planning"
    },
    {
        "assignee": "최마케",
        "description": "Q2 콘텐츠 마케팅 전략서 준비",
        "deadline": "2026-04-20",
        "category": "marketing"
    },
    {
        "assignee": "정디자",
        "description": "검색 필터 UI 리디자인 시안 3종 제출",
        "deadline": "2026-04-18",
        "category": "design"
    },
    {
        "assignee": "이개발",
        "description": "검색 성능 벤치마크 보고서 공유",
        "deadline": "2026-04-08",
        "category": "development"
    }
]

print(f"기대 액션 아이템: {len(expected_items)}개")
for i, item in enumerate(expected_items, 1):
    print(f"  {i}. [{item['assignee']}] {item['description']} (기한: {item['deadline']})")

In [ ]:
# ==========================================
# 평가 함수
# ==========================================
def evaluate_extraction(extracted: list[dict], expected: list[dict]) -> dict:
    """추출 결과를 기대 결과와 비교하여 평가합니다.
    
    매칭 기준: 담당자 이름 일치 + 기한 일치
    """
    matched = 0
    match_details = []
    
    for exp in expected:
        found = False
        for ext in extracted:
            assignee_match = exp["assignee"] in ext.get("assignee", "")
            deadline_match = exp["deadline"] == ext.get("deadline", "")
            
            if assignee_match and deadline_match:
                matched += 1
                found = True
                match_details.append({
                    "expected": exp["description"],
                    "extracted": ext.get("description", ""),
                    "status": "MATCH"
                })
                break
        
        if not found:
            match_details.append({
                "expected": exp["description"],
                "extracted": "-",
                "status": "MISS"
            })
    
    # 지표 계산
    recall = matched / len(expected) if expected else 0
    precision = matched / len(extracted) if extracted else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )
    
    return {
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "matched": matched,
        "total_expected": len(expected),
        "total_extracted": len(extracted),
        "details": match_details
    }

In [ ]:
# ==========================================
# 평가 실행
# ==========================================
eval_result = evaluate_extraction(
    extracted=final_state["action_items"],
    expected=expected_items
)

print("=== 품질 평가 결과 ===\n")
print(f"  Precision: {eval_result['precision']:.1%} ({eval_result['matched']}/{eval_result['total_extracted']})")
print(f"  Recall:    {eval_result['recall']:.1%} ({eval_result['matched']}/{eval_result['total_expected']})")
print(f"  F1 Score:  {eval_result['f1']:.1%}")

print("\n--- 매칭 상세 ---")
for d in eval_result["details"]:
    status_icon = "OK" if d["status"] == "MATCH" else "MISS"
    print(f"  [{status_icon}] 기대: {d['expected'][:40]}")
    if d["status"] == "MATCH":
        print(f"         추출: {d['extracted'][:40]}")

# 품질 기준 확인
print("\n--- 품질 기준 충족 여부 ---")
thresholds = {"precision": 0.9, "recall": 0.8, "f1": 0.85}
for metric, threshold in thresholds.items():
    value = eval_result[metric]
    passed = value >= threshold
    status = "PASS" if passed else "FAIL"
    print(f"  {metric}: {value:.1%} (기준: {threshold:.0%}) → {status}")

## 섹션 10. Langfuse 통합

파이프라인 실행을 Langfuse로 추적하여 각 에이전트의 실행 시간, 토큰 사용량, 비용을 모니터링합니다.

In [ ]:
# ==========================================
# Langfuse 콜백 설정
# ==========================================
if LANGFUSE_ENABLED:
    from langfuse.callback import CallbackHandler

    langfuse_handler = CallbackHandler(
        public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
        secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
        host=os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
    )
    print("Langfuse CallbackHandler 초기화 완료")
    print(f"호스트: {os.getenv('LANGFUSE_HOST', 'https://cloud.langfuse.com')}")
else:
    langfuse_handler = None
    print("Langfuse 비활성화 (API 키 미설정)")
    print("활성화하려면 .env 파일에 LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY를 설정하세요.")

In [ ]:
# ==========================================
# Langfuse 추적이 포함된 파이프라인 실행
# ==========================================
if LANGFUSE_ENABLED:
    print("Langfuse 추적이 포함된 파이프라인 실행 시작...\n")
    
    # Mock 저장소 초기화
    _jira_tickets.clear()
    _notion_pages.clear()
    _slack_messages.clear()
    
    traced_result = pipeline.invoke(
        initial_state,
        config={"callbacks": [langfuse_handler]}
    )
    
    print("\nLangfuse 추적 완료!")
    print("Langfuse 대시보드에서 트레이스를 확인하세요.")
    print(f"추출된 액션 아이템: {len(traced_result['action_items'])}개")
    print(f"Jira 티켓: {len(traced_result['jira_tickets'])}건")
else:
    print("Langfuse 비활성화 상태 — 이전 실행 결과를 사용합니다.")
    print("Langfuse 통합을 테스트하려면 API 키를 설정한 후 이 셀을 다시 실행하세요.")

## 섹션 11. Exercise

아래 두 가지 과제를 직접 구현해 보세요.

### Exercise 1: 후속 회의 알림 에이전트

액션 아이템의 기한이 다가오면 Slack으로 리마인더를 보내는 에이전트를 추가하세요.

**요구사항**:
1. `MeetingState`에 `reminders` 필드를 추가
2. 기한 3일 전에 리마인더 생성 로직 구현
3. LangGraph에 `reminder` 노드를 추가
4. Mock Slack MCP로 알림 전송

In [ ]:
# ==========================================
# Exercise 1: 후속 회의 알림 에이전트
# ==========================================

# TODO: 기한 임박 아이템 필터링 함수
def check_deadlines(action_items: list[dict], days_before: int = 3) -> list[dict]:
    """기한이 days_before일 이내인 액션 아이템을 반환합니다."""
    # 힌트: datetime.strptime(item["deadline"], "%Y-%m-%d")으로 날짜 파싱
    # 힌트: (deadline - today).days로 남은 일수 계산
    pass


# TODO: 리마인더 노드 함수
def send_reminders(state: MeetingState) -> dict:
    """기한 임박 아이템에 대해 Slack 리마인더를 전송합니다."""
    # 힌트: check_deadlines()로 임박 아이템 필터
    # 힌트: send_message()로 각 담당자에게 알림 전송
    pass


# TODO: LangGraph에 reminder 노드 추가
# 힌트: graph.add_node("reminder", send_reminders)
# 힌트: registrar → reminder → END로 엣지 수정

print("Exercise 1: TODO 구현을 완료하세요.")

### Exercise 2: 회의 유형별 프롬프트 최적화

회의 유형(브레인스토밍, 의사결정, 스프린트 리뷰)에 따라 다른 추출 프롬프트를 적용하세요.

**요구사항**:
1. 회의 유형 자동 분류 에이전트 추가
2. 유형별 전용 프롬프트 3개 작성
3. 분류 결과에 따라 적절한 프롬프트 선택
4. 각 유형별 추출 정확도 비교 평가

In [ ]:
# ==========================================
# Exercise 2: 회의 유형별 프롬프트 최적화
# ==========================================

# TODO: 회의 유형 정의
MEETING_TYPES = {
    "brainstorming": {
        "description": "아이디어 발산 회의",
        "prompt": """아이디어와 제안을 중심으로 추출하세요.
        각 아이디어의 제안자, 구체적 내용, 후속 검토 기한을 포함합니다.
        우선순위는 실현 가능성과 영향도를 기준으로 판단합니다."""
    },
    "decision": {
        "description": "의사결정 회의",
        "prompt": """결정 사항과 후속 조치를 중심으로 추출하세요.
        각 결정의 실행 담당자, 구체적 실행 내용, 완료 기한을 포함합니다.
        우선순위는 비즈니스 임팩트를 기준으로 판단합니다."""
    },
    "sprint_review": {
        "description": "스프린트 리뷰 회의",
        "prompt": """완료 항목과 다음 스프린트 과제를 중심으로 추출하세요.
        미완료 항목의 담당자, 잔여 작업 내용, 다음 스프린트 기한을 포함합니다.
        우선순위는 스프린트 목표 기여도를 기준으로 판단합니다."""
    }
}


# TODO: 회의 유형 분류 함수
def classify_meeting(transcript: str) -> str:
    """회의록을 분석하여 유형을 분류합니다."""
    # 힌트: LLM에게 회의 유형을 분류하도록 프롬프트 작성
    # 힌트: 반환값은 "brainstorming", "decision", "sprint_review" 중 하나
    pass


# TODO: 유형별 추출 정확도 비교 함수
def compare_by_type(transcript: str, expected: list[dict]) -> dict:
    """각 유형별 프롬프트로 추출한 결과의 정확도를 비교합니다."""
    # 힌트: 각 MEETING_TYPES의 프롬프트로 추출 실행
    # 힌트: evaluate_extraction()으로 각각 평가
    # 힌트: F1 Score가 가장 높은 유형을 반환
    pass


print("Exercise 2: TODO 구현을 완료하세요.")

## 마무리

### 오늘 배운 것

1. **Whisper API**로 한국어 음성을 텍스트로 변환하고, 텍스트 폴백을 구현했습니다
2. **Pydantic 구조화 출력**으로 회의록에서 액션 아이템(담당자, 업무, 기한, 우선순위)을 정형 데이터로 추출했습니다
3. **FastMCP Mock 서버**로 Jira/Notion/Slack 연동을 안전하게 시뮬레이션했습니다
4. **LangGraph 멀티에이전트 파이프라인**으로 요약→추출→등록 전체 흐름을 자동화했습니다
5. **품질 평가**로 Precision/Recall/F1 Score를 측정하고 품질 기준 충족 여부를 확인했습니다

### 핵심 기술 연결

| 이번 에피소드 | 참고 에피소드 |
|-------------|-------------|
| Pydantic 구조화 추출 | **EP08** JSON & Function Calling |
| LangGraph 파이프라인 | **EP10** 멀티에이전트 시스템 |
| FastMCP Mock 서버 | **EP12** MCP 서버 직접 만들기 |
| 품질 평가 (F1 Score) | **EP04** Agent Harness |
| Langfuse 모니터링 | **EP06** 디버깅 |

### 다음 에피소드

**EP23. 기술 문서 챗봇** -- RAG + 에이전트로 사내 기술 문서를 대화형 인터페이스로 제공합니다.